# 🔄 Re-Ranking: Improving Retrieval Quality with Two-Stage Filtering

**Course Reference:** [Ultimate RAG Bootcamp Using Langchain, LangGraph & Langsmith](https://www.udemy.com/course/ultimate-rag-bootcamp-using-langchainlanggraph-langsmith)

---

## 📚 Learning Objectives

By the end of this notebook, you will understand:
1. What **Re-ranking** is and why it's essential for high-quality retrieval
2. The **two-stage retrieval** architecture (fast retrieval → accurate re-ranking)
3. How to use **LLMs as re-rankers** to score document relevance
4. How to implement a complete re-ranking pipeline with LangChain

---

## 🧠 Key Concepts

### What is Re-Ranking?

Re-ranking is a **second-stage filtering process** that refines the results from an initial retrieval step:

```
Stage 1 (Fast):    Query → Fast Retriever (BM25/FAISS) → Top-K Documents (rough candidates)
                                    ↓
Stage 2 (Accurate): Top-K Documents → Re-ranker Model → Re-ordered Documents (best at top)
```

### Why Use Re-Ranking?

| Stage | Method | Speed | Accuracy |
|-------|--------|-------|----------|
| **Stage 1** | Vector similarity / BM25 | ⚡ Very Fast | 🔶 Good |
| **Stage 2** | Cross-encoder / LLM | 🐢 Slower | ✅ Excellent |

### Types of Re-Rankers

1. **Cross-Encoders**: Neural models that score query-document pairs together
2. **LLM-based Re-rankers**: Use LLMs to judge document relevance
3. **Cohere Rerank**: Commercial API for high-quality re-ranking
4. **ColBERT**: Late interaction models for efficient re-ranking

---

## 📦 Step 1: Understanding the Re-Ranking Process

In this notebook, we'll implement **LLM-based re-ranking**, where we:
1. Use a fast retriever (FAISS) to get initial candidates
2. Ask an LLM to rank these candidates by relevance
3. Reorder documents based on LLM's judgment

This approach leverages the LLM's deep understanding of language and context to produce more accurate rankings than simple vector similarity.

### 💡 Why Re-Ranking Matters

Re-ranking is a second-stage filtering process in retrieval systems, especially in RAG pipelines, where we:

1. **First Stage (Fast Retrieval):** Use a fast retriever (like BM25, FAISS, hybrid) to fetch top-k documents quickly.
   - ✅ Pros: Very fast, can scan millions of documents
   - ❌ Cons: May not perfectly understand query intent

2. **Second Stage (Accurate Re-ranking):** Use a more accurate but slower model (like a cross-encoder or LLM) to re-score and reorder those documents by relevance to the query.
   - ✅ Pros: Deep understanding of query-document relevance
   - ❌ Cons: Too slow to run on entire corpus

👉 **Key Insight:** It ensures that the most relevant documents appear at the top, significantly improving the final answer from the LLM.

### 🎯 When to Use Re-Ranking

- When initial retrieval returns many somewhat-relevant documents
- When precision at the top positions matters (e.g., only using top 3 docs)
- When the LLM context window is limited and you need the BEST documents
- When dealing with complex queries that require semantic understanding

In [3]:
# ============================================================================
# STEP 2: IMPORT REQUIRED LIBRARIES
# ============================================================================

# TextLoader: Loads plain text files into LangChain Document format
from langchain.document_loaders import TextLoader

# RecursiveCharacterTextSplitter: Intelligently splits text into chunks
# It tries to keep paragraphs, sentences, and words together when possible
from langchain.text_splitter import RecursiveCharacterTextSplitter

# init_chat_model: Universal initializer for various chat models (OpenAI, Groq, etc.)
from langchain.chat_models import init_chat_model

# PromptTemplate: Creates reusable prompt templates with variable substitution
from langchain.prompts import PromptTemplate

# Document: LangChain's standard document class with page_content and metadata
from langchain.schema import Document

# StrOutputParser: Extracts the string content from LLM responses
from langchain_core.output_parsers import StrOutputParser

print("✅ All libraries imported successfully!")

/Users/sourav.banerjee/Documents/Codebases/2. AI ENGINEERING/RAG_Demystified/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All libraries imported successfully!


In [4]:
# ============================================================================
# STEP 3: LOAD AND CHUNK THE DOCUMENTS
# ============================================================================
# This step prepares our knowledge base for retrieval

# Load text file containing information about LangChain
# In production, this could be PDFs, web pages, databases, etc.
loader = TextLoader("langchain_sample.txt")
raw_docs = loader.load()

print(f"📄 Loaded {len(raw_docs)} document(s)")
print(f"   Total characters: {len(raw_docs[0].page_content)}")

# Split text into smaller, manageable chunks
# Parameters explained:
# - chunk_size=500: Maximum characters per chunk (adjust based on your embedding model)
# - chunk_overlap=50: Characters shared between adjacent chunks (preserves context)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(raw_docs)

print(f"✅ Split into {len(docs)} chunks")
print(f"\n📋 Sample chunk preview:")
print("-" * 50)
print(docs[0].page_content[:200] + "...")
docs


📄 Loaded 1 document(s)
   Total characters: 1980
✅ Split into 6 chunks

📋 Sample chunk preview:
--------------------------------------------------
LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes compo...


[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.\nBM25 is a traditional 

In [5]:
# ============================================================================
# STEP 4: DEFINE THE USER QUERY
# ============================================================================
# This is the question we want to answer using our RAG pipeline
# The re-ranker will help find the MOST relevant documents for this query

query = "How can I use langchain to build an application with memory and tools?"

print(f"🔍 User Query: \"{query}\"")
print("\n💡 This query tests the re-ranker's ability to find documents about:")
print("   - LangChain framework")
print("   - Memory capabilities") 
print("   - Tool integration")
print("   - Application development")

🔍 User Query: "How can I use langchain to build an application with memory and tools?"

💡 This query tests the re-ranker's ability to find documents about:
   - LangChain framework
   - Memory capabilities
   - Tool integration
   - Application development


In [6]:
# ============================================================================
# STEP 5: CREATE FAST RETRIEVER (Stage 1 - Initial Retrieval)
# ============================================================================
# We use FAISS + HuggingFace embeddings for fast initial retrieval
# This retrieves MORE documents than we need (k=8), then re-rank to get the best ones

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize embedding model
# "all-MiniLM-L6-v2" is a lightweight model with 384-dim embeddings
# - Fast inference speed
# - Good quality for general semantic similarity
# - ~22M parameters (small enough for CPU)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create FAISS vector store
# FAISS (Facebook AI Similarity Search) provides efficient similarity search
vectorstore = FAISS.from_documents(docs, embedding_model)

# Create retriever that returns top-8 documents
# We retrieve MORE than we need, then re-rank to find the best ones
# This is the "retrieve many, keep few" strategy
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

print("✅ FAISS retriever created!")
print(f"   - Embedding model: all-MiniLM-L6-v2")
print(f"   - Top-k retrieval: 8 documents")
print("   - These will be re-ranked to find the most relevant ones")

✅ FAISS retriever created!
   - Embedding model: all-MiniLM-L6-v2
   - Top-k retrieval: 8 documents
   - These will be re-ranked to find the most relevant ones


In [7]:
# ============================================================================
# ALTERNATIVE: CREATE RETRIEVER WITH OPENAI EMBEDDINGS
# ============================================================================
# OpenAI embeddings often provide better quality but require an API key
# This is shown as an alternative to the HuggingFace embeddings above

import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Set OpenAI API key from environment
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

from langchain_openai import OpenAIEmbeddings

# OpenAI's text-embedding-ada-002 (default) produces 1536-dim embeddings
# - Higher quality than most open-source models
# - Requires API key and incurs costs
# - ~$0.0001 per 1K tokens
embeddings = OpenAIEmbeddings()

# Create alternative vector store with OpenAI embeddings
vectorstore_openai = FAISS.from_documents(docs, embeddings)
# ============================================================================
# INSPECT THE OPENAI RETRIEVER
# ============================================================================
# Compare with the OpenAI retriever - same structure, different embedding model

print("📋 OpenAI-based Retriever Configuration:")
retriever_openai = vectorstore_openai.as_retriever(search_kwargs={"k": 8})

print("✅ OpenAI embeddings retriever created!")
print("   - Model: text-embedding-ada-002 (1536-dim)")
print("   - Higher quality, but requires API key")

📋 OpenAI-based Retriever Configuration:
✅ OpenAI embeddings retriever created!
   - Model: text-embedding-ada-002 (1536-dim)
   - Higher quality, but requires API key


In [8]:
# Inspect the HuggingFace-based retriever configuration
print("🔍 HuggingFace Retriever Configuration:")
retriever

🔍 HuggingFace Retriever Configuration:


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x353ef6050>, search_kwargs={'k': 8})

In [9]:
# Inspect the OpenAI-based retriever configuration
print("🔍 OpenAI Retriever Configuration:")
retriever_openai

🔍 OpenAI Retriever Configuration:


VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x3aa6a06d0>, search_kwargs={'k': 8})

In [10]:
# ============================================================================
# STEP 6: INITIALIZE THE LLM FOR RE-RANKING
# ============================================================================
# We use an LLM to re-rank documents by asking it to judge relevance
# Groq provides fast, free inference for open-source models

from langchain.chat_models import init_chat_model

# Set Groq API key for free LLM access
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Initialize Llama 3.1 8B via Groq
# - "groq:llama-3.1-8b-instant": Fast inference on Groq's infrastructure
# - 8B parameters: Good balance of speed and capability for re-ranking
# - Free tier available for experimentation
llm = init_chat_model("groq:llama-3.1-8b-instant")

print("✅ LLM initialized for re-ranking!")
print("   - Model: Llama 3.1 8B Instant (via Groq)")
print("   - This LLM will judge document relevance")
llm

✅ LLM initialized for re-ranking!
   - Model: Llama 3.1 8B Instant (via Groq)
   - This LLM will judge document relevance


ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3aa6bdb90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x3ab3bdb50>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [11]:
# ============================================================================
# STEP 7: CREATE THE RE-RANKING PROMPT
# ============================================================================
# This prompt instructs the LLM to act as a re-ranker
# It takes a question and a list of documents, then returns ranked indices

# The key to effective LLM-based re-ranking:
# 1. Clear instructions on the task (ranking, not answering)
# 2. Numbered documents for easy reference
# 3. Specific output format for easy parsing

prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's question.
- Consider semantic meaning, not just keyword matches.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)

IMPORTANT: Only output the comma-separated indices, nothing else.
""")

print("✅ Re-ranking prompt template created!")
print("\n📝 The LLM will:")
print("   1. Receive a question and numbered documents")
print("   2. Judge relevance of each document")
print("   3. Return indices ordered by relevance")

✅ Re-ranking prompt template created!

📝 The LLM will:
   1. Receive a question and numbered documents
   2. Judge relevance of each document
   3. Return indices ordered by relevance


In [12]:
# ============================================================================
# STEP 8: STAGE 1 - FAST RETRIEVAL
# ============================================================================
# This is the first stage of our two-stage retrieval process
# We retrieve 8 documents using fast vector similarity search

retrieved_docs = retriever.invoke(query)

print(f"📥 Stage 1 Complete: Retrieved {len(retrieved_docs)} documents")
print("=" * 60)
print("\n📋 Retrieved documents (before re-ranking):")
for i, doc in enumerate(retrieved_docs):
    preview = doc.page_content[:100].replace('\n', ' ')
    print(f"\n[{i+1}] {preview}...")

print("\n⚠️ Note: The order is based on vector similarity, which may not be optimal")
retrieved_docs

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


📥 Stage 1 Complete: Retrieved 6 documents

📋 Retrieved documents (before re-ranking):

[1] LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to in...

[2] LangChain is a flexible framework designed for developing applications powered by large language mod...

[3] LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This e...

[4] FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spa...

[5] Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved a...

[6] Dense retrieval uses embeddings to match query and documents in a vector space. This allows capturin...

⚠️ Note: The order is based on vector similarity, which may not be optimal


[Document(id='2805cfd3-2fae-4d7e-85d5-e5cdaa364f4a', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='52b2a911-bfd0-4e84-bc20-7f016d901d05', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='efae8014-882e-4d0d-8572-ffc18d648b87', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging F

In [13]:
# ============================================================================
# STEP 9: CREATE THE RE-RANKING CHAIN
# ============================================================================
# Using LangChain Expression Language (LCEL) to create a chain
# Chain: prompt → LLM → parse string output

# The chain processes:
# 1. prompt: Formats the template with question and documents
# 2. llm: Sends to LLM and gets response
# 3. StrOutputParser: Extracts just the string content

chain = prompt | llm | StrOutputParser()

print("✅ Re-ranking chain created!")
print("   Flow: prompt → LLM → StrOutputParser")
chain

✅ Re-ranking chain created!
   Flow: prompt → LLM → StrOutputParser


PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n- Think about the relevance of each document to the user\'s question.\n- Consider semantic meaning, not just keyword matches.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput format: comma-separated document indices (e.g., 2,1,3,0,...)\n\nIMPORTANT: Only output the comma-separated indices, nothing else.\n')
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3aa6bdb90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x3ab3bdb50>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser()

In [14]:
# ============================================================================
# STEP 10: FORMAT DOCUMENTS FOR THE LLM
# ============================================================================
# We need to convert Document objects into a numbered string format
# that the LLM can easily read and reference in its ranking.
#
# 🧠 Why this formatting matters:
# - Numbers (1, 2, 3...) give the LLM clear identifiers to reference
# - Plain text is easier for the LLM to process than Python objects
# - Consistent formatting helps the LLM produce reliable output

# Create numbered list of document contents (1-indexed for human readability)
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]

# Join into a single string with newlines between documents
formatted_docs = "\n".join(doc_lines)

print("📝 Documents formatted for LLM input:")
print(f"   - Total documents: {len(doc_lines)}")
print(f"   - Total characters: {len(formatted_docs)}")
print("\n💡 Each document is numbered so the LLM can reference them in its ranking")

📝 Documents formatted for LLM input:
   - Total documents: 6
   - Total characters: 1997

💡 Each document is numbered so the LLM can reference them in its ranking


In [16]:
# View the individual document lines (each will be shown to the LLM)
print("📋 Individual document entries:")
for i, line in enumerate(doc_lines):
    print(f"\n--- Document {i+1} ---")
    print(line[:200] + "..." if len(line) > 200 else line)

📋 Individual document entries:

--- Document 1 ---
1. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain ...

--- Document 2 ---
2. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes co...

--- Document 3 ---
3. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use c...

--- Document 4 ---
4. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed indexes, which makes it scalable for large document sto...

--- Document 5 ---
5. Retrieval-Augmented Generation (RAG) is a powerful

In [17]:
# View the complete formatted document string that will be sent to LLM
print("📝 Complete formatted documents string:")
print("=" * 60)
print(formatted_docs)

📝 Complete formatted documents string:
1. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.
2. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.
3. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.
4. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It sup

In [18]:
# ============================================================================
# STEP 11: STAGE 2 - LLM RE-RANKING
# ============================================================================
# This is where the magic happens! The LLM analyzes all documents
# and returns them ordered by relevance to the query

print("🔄 Sending documents to LLM for re-ranking...")
print(f"   Query: \"{query}\"")
print(f"   Documents to rank: {len(retrieved_docs)}")
print()

# Invoke the re-ranking chain
response = chain.invoke({"question": query, "documents": formatted_docs})

print("✅ LLM Re-ranking complete!")
print(f"\n📊 LLM's ranking response: {response}")
print("\n💡 These indices represent the LLM's judgment of relevance")
response

🔄 Sending documents to LLM for re-ranking...
   Query: "How can I use langchain to build an application with memory and tools?"
   Documents to rank: 6

✅ LLM Re-ranking complete!

📊 LLM's ranking response: 2,1,3

💡 These indices represent the LLM's judgment of relevance


'2,1,3'

In [19]:
# ============================================================================
# STEP 12: PARSE THE LLM'S RANKING RESPONSE
# ============================================================================
# The LLM returns comma-separated indices (1-indexed)
# We need to parse these and convert to 0-indexed for Python

# Parse the response:
# 1. Split by comma
# 2. Strip whitespace
# 3. Filter for valid digits
# 4. Convert to 0-indexed (subtract 1)
indices = [int(x.strip()) - 1 for x in response.split(",") if x.strip().isdigit()]

print(f"📊 Parsed ranking indices (0-indexed): {indices}")
print(f"\n   Interpretation:")
print(f"   - Most relevant document: index {indices[0]} (originally position {indices[0]+1})")
print(f"   - Least relevant document: index {indices[-1]} (originally position {indices[-1]+1})")
indices

📊 Parsed ranking indices (0-indexed): [1, 0, 2]

   Interpretation:
   - Most relevant document: index 1 (originally position 2)
   - Least relevant document: index 2 (originally position 3)


[1, 0, 2]

In [22]:
# View the original retrieved documents (before re-ranking)
print("📋 Original retrieved documents (pre-reranking order):")
print("=" * 60)
for i, doc in enumerate(retrieved_docs):
    print(f"\n[{i}] {doc.page_content[:150]}...")

📋 Original retrieved documents (pre-reranking order):

[0] LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accu...

[1] LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to ...

[2] LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different mod...

[3] FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed indexes,...

[4] Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses....

[5] Dense retrieval uses embeddings to match query and documents in a vector space. This allows capturing semantic meaning, making it useful for fuzzy

In [23]:
# ============================================================================
# EXAMPLE: INSPECTING A SPECIFIC DOCUMENT
# ============================================================================
# Documents are stored in a list, so we can access them by index (0-based)
# Here we examine document at index 5 (the 6th document in the list)
# This helps verify the re-ranking is working correctly

print("📄 Document at index 5 (6th document):")
print("-" * 60)

if len(retrieved_docs) > 5:
    print(retrieved_docs[5].page_content)
    print("-" * 60)
    print(f"\n📎 Metadata: {retrieved_docs[5].metadata}")
else:
    print(f"Only {len(retrieved_docs)} documents available")

📄 Document at index 5 (6th document):
------------------------------------------------------------
Dense retrieval uses embeddings to match query and documents in a vector space. This allows capturing semantic meaning, making it useful for fuzzy or natural language queries.
LangChain supports hybrid retrieval by combining BM25 and dense similarity scores. This approach improves both precision and recall in document search.
------------------------------------------------------------

📎 Metadata: {'source': 'langchain_sample.txt'}


In [24]:
# ============================================================================
# STEP 13: REORDER DOCUMENTS BASED ON LLM RANKING
# ============================================================================
# Now we reorder our documents according to the LLM's judgment
# Documents are rearranged so the most relevant appear first

# Reorder documents using the parsed indices
# Safety check: only use valid indices within range
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i < len(retrieved_docs)]

print(f"✅ Re-ranking complete!")
print(f"   Original documents: {len(retrieved_docs)}")
print(f"   Re-ranked documents: {len(reranked_docs)}")
print("\n📊 Document order changed from vector similarity → LLM judgment")
reranked_docs

✅ Re-ranking complete!
   Original documents: 6
   Re-ranked documents: 3

📊 Document order changed from vector similarity → LLM judgment


[Document(id='52b2a911-bfd0-4e84-bc20-7f016d901d05', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='2805cfd3-2fae-4d7e-85d5-e5cdaa364f4a', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='efae8014-882e-4d0d-8572-ffc18d648b87', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging F

In [25]:
# ============================================================================
# STEP 14: DISPLAY FINAL RE-RANKED RESULTS
# ============================================================================
# The re-ranked documents now have the most relevant at the top
# This significantly improves the context quality for the LLM in RAG

print("=" * 70)
print("📊 FINAL RE-RANKED RESULTS (Best → Worst)")
print("=" * 70)

for i, doc in enumerate(reranked_docs, 1):
    print(f"\n🏆 Rank {i}:")
    print("-" * 50)
    print(doc.page_content)

print("\n" + "=" * 70)
print("✅ RE-RANKING COMPLETE!")
print("=" * 70)
print("""
💡 KEY TAKEAWAYS:

1. Stage 1 (Fast Retrieval): Retrieved 8 documents using FAISS vector similarity
   - Fast but may not perfectly order by relevance

2. Stage 2 (LLM Re-ranking): LLM analyzed and reordered documents
   - Slower but understands query intent deeply
   - Places most relevant documents at top

3. Benefits:
   - Better context for answer generation
   - More accurate final responses
   - Improved user experience

4. Trade-offs:
   - Additional latency from LLM call
   - Extra API costs
   - Best used when precision matters
""")

📊 FINAL RE-RANKED RESULTS (Best → Worst)

🏆 Rank 1:
--------------------------------------------------
LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.

🏆 Rank 2:
--------------------------------------------------
LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.

🏆 Rank 3:
--------------------------------------------------
LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific u